[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C06_Interpretability_Course/05_superposition_sae/05_superposition_sae.ipynb)

# 05 · Superposition 与 SAE：把叠加的特征拆回来

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy + matplotlib，手写全部梯度与 Adam，全程 CPU 一两分钟内跑完。

**本 notebook 你将完成：**

1. 复现 **Toy Models of Superposition**（Elhage 2022）最小版：$n=20$ 个稀疏特征 × $d=5$ 维瓶颈的线性 autoencoder（$\hat x = \mathrm{ReLU}(W^\top W x + b)$），手写梯度训练；
2. 画 $W^\top W$ 热图与特征范数图，亲眼看到**稠密 → PCA、稀疏 → 叠加**的相变；
3. 在叠加表示上**从零训练一个 SAE**（encoder ReLU + decoder，手写反向传播：MSE + λ·L1、decoder 行归一化、手写 Adam）；
4. 用**余弦相似度匹配矩阵**验证 SAE 字典方向与真特征方向一一对应（真模型里没有这种 ground truth 奢侈，沙盒里有）；
5. 统计 **L0 / 重建 $R^2$ / dead features**，并做 **λ 扫描**画出 L0–重建帕累托权衡曲线；
6. 完成 4 道 ✏️ 练习（SAE 前向 + loss / 手写梯度 vs 数值梯度对拍 / 贪心特征–字典匹配 / dead feature 检测与 resampling）。

参考文献：Elhage et al. 2022 *Toy Models of Superposition* (transformer-circuits.pub/2022/toy_model) · Bricken et al. 2023 *Towards Monosemanticity* (transformer-circuits.pub/2023/monosemantic-features) · Templeton et al. 2024 *Scaling Monosemanticity* (transformer-circuits.pub/2024/scaling-monosemanticity) · Gao et al. 2024 (arXiv:2406.04093)

## 1 · 复现 Toy Models of Superposition（最小版）

[Elhage 2022] 的玩具模型把"叠加"变成可以完整观察的对象（讲解页 §2）：

- **数据**：$x \in \mathbb{R}^{20}$，每个特征以概率 $p$ 激活（激活值 $\sim U[0,1]$），否则为 0；稀疏度 $S = 1-p$；
- **模型**：线性瓶颈 autoencoder，$h = Wx \in \mathbb{R}^{5}$，$\hat{x} = \mathrm{ReLU}(W^\top W x + b)$，$W \in \mathbb{R}^{5\times 20}$，列 $W_i$ 就是特征 $i$ 的表示方向；
- **损失**：重要性加权 MSE，$\mathcal L = \tfrac{1}{B}\sum_b \sum_i I_i\,(x_i - \hat x_i)^2$，$I_i = 0.9^{\,i}$ 递减。

模型只有 5 维可用却要重建 20 个特征——它必须取舍。手写梯度只有三条链（记 $H = XW^\top$、$Z = HW + b$、$\hat X = \mathrm{ReLU}(Z)$）：

$$\frac{\partial \mathcal L}{\partial Z} = \tfrac{2}{B}\, I \odot (\hat X - X)\odot \mathbb{1}[Z>0],\qquad
\frac{\partial \mathcal L}{\partial W} = \underbrace{H^\top \tfrac{\partial \mathcal L}{\partial Z}}_{\text{decoder 路径}} \;+\; \underbrace{\Big(\tfrac{\partial \mathcal L}{\partial Z}\, W^\top\Big)^{\!\top} X}_{\text{encoder 路径}},\qquad
\frac{\partial \mathcal L}{\partial b} = \textstyle\sum_b \tfrac{\partial \mathcal L}{\partial Z}$$

（$W$ 同时出现在 encode 与 decode 两处，梯度是两条路径之和。）我们在两种激活概率下各训一个模型：**稠密** $p=0.8$ 与**稀疏** $p=0.05$，预期出现相变：稠密 → 只表示最重要的约 $d$ 个特征（PCA 模式）；稀疏 → 把远多于 $d$ 个特征以近正交方向叠加塞进 5 维（收益 $O(p)$ vs 干涉代价 $O(p^2)$，见讲解页 §2.1）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- 手写 Adam（toy model 与 SAE 共用）----
def adam_init(params):
    return {k: [np.zeros_like(v), np.zeros_like(v)] for k, v in params.items()}

def adam_step(params, grads, state, lr, t, b1=0.9, b2=0.999, eps=1e-8):
    for k in params:
        m, v = state[k]
        m[...] = b1 * m + (1 - b1) * grads[k]
        v[...] = b2 * v + (1 - b2) * grads[k] ** 2
        params[k] -= lr * (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + eps)

def relu(z):
    return np.maximum(z, 0.0)

def sample_features(B, n, p, rng):
    "稀疏特征向量：每个特征以概率 p 激活，激活值 ~ U[0,1]，否则为 0。"
    return (rng.random((B, n)) < p) * rng.random((B, n))

N_FEAT, D_HID = 20, 5
IMPORTANCE = 0.9 ** np.arange(N_FEAT)          # I_i = 0.9^i，重要性递减

def train_toy(p_active, steps=6000, batch=1024, lr=3e-3, seed=0):
    "训练 x_hat = ReLU(W^T W x + b)，重要性加权 MSE，手写梯度 + Adam。"
    rng = np.random.default_rng(seed)
    params = {"W": rng.normal(size=(D_HID, N_FEAT)) / np.sqrt(N_FEAT),
              "b": np.zeros(N_FEAT)}
    state = adam_init(params)
    for t in range(1, steps + 1):
        X = sample_features(batch, N_FEAT, p_active, rng)
        H = X @ params["W"].T                       # (B,d)   encode: h = Wx
        Z = H @ params["W"] + params["b"]           # (B,n)   decode 预激活
        Xhat = relu(Z)
        dZ = 2 * IMPORTANCE * (Xhat - X) / batch * (Z > 0)
        grads = {"W": H.T @ dZ + (dZ @ params["W"].T).T @ X,   # decoder 路径 + encoder 路径
                 "b": dZ.sum(0)}
        adam_step(params, grads, state, lr, t)
    loss = float((IMPORTANCE * (Xhat - X) ** 2).sum(1).mean())
    return params["W"], params["b"], loss

W_dense,  b_dense,  loss_dense  = train_toy(p_active=0.80)   # 稠密：S = 0.20
W_sparse, b_sparse, loss_sparse = train_toy(p_active=0.05)   # 稀疏：S = 0.95

norms_dense  = np.linalg.norm(W_dense,  axis=0)
norms_sparse = np.linalg.norm(W_sparse, axis=0)
n_repr_dense  = int((norms_dense  > 0.5).sum())
n_repr_sparse = int((norms_sparse > 0.5).sum())
print(f"dense  (p=0.80): loss={loss_dense:.4f}  被表示特征数 (‖W_i‖>0.5) = {n_repr_dense}/{N_FEAT}")
print(f"sparse (p=0.05): loss={loss_sparse:.4f}  被表示特征数 (‖W_i‖>0.5) = {n_repr_sparse}/{N_FEAT}")
assert n_repr_dense <= 8,   "稠密情形应只表示最重要的约 d 个特征（PCA 模式）"
assert n_repr_sparse >= 12, "稀疏情形应把远多于 d 个特征叠加进 5 维"


In [ ]:
# 相变可视化：Gram 矩阵 W^T W（对角 = ‖W_i‖²，非对角 = 特征间干涉）+ 特征范数
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5),
                         gridspec_kw={"height_ratios": [3, 1.3]})
for col, (W, p, n_r) in enumerate([(W_dense, 0.80, n_repr_dense),
                                   (W_sparse, 0.05, n_repr_sparse)]):
    G = W.T @ W
    im = axes[0, col].imshow(G, cmap="RdBu_r", vmin=-1, vmax=1)
    axes[0, col].set_title(f"$W^\\top W$   p={p}  ({n_r}/{N_FEAT} features represented)")
    axes[0, col].set_xlabel("feature j"); axes[0, col].set_ylabel("feature i")
    fig.colorbar(im, ax=axes[0, col], fraction=0.046)
    norms = np.linalg.norm(W, axis=0)
    axes[1, col].bar(np.arange(N_FEAT), norms, color=plt.cm.viridis(IMPORTANCE))
    axes[1, col].axhline(1.0, color="gray", lw=0.8, ls="--")
    axes[1, col].set_xlabel("feature index (importance decreasing)")
    axes[1, col].set_ylabel(r"$\|W_i\|$")
plt.tight_layout(); plt.show()

# 读图：左（稠密）= 左上角近似单位阵、其余特征范数≈0 —— 重要特征独占正交维度；
# 右（稀疏）= 几乎所有对角元非零 + 成片小幅非对角干涉 —— 特征叠加成近正交方向。

## 2 · 从零训练 SAE：把叠加拆回来

稀疏 toy model 的瓶颈激活 $a = Wx \in \mathbb{R}^5$ 是约 20 个特征方向的稀疏线性叠加——而我们**知道全部真相**（真字典 = $W$ 的列）。现在假装不知道，用 SAE 反演（讲解页 §3）：

$$h = \mathrm{ReLU}(W_e\, a + b_e) \in \mathbb{R}^{40},\qquad
\hat a = W_d^\top h + b_d \in \mathbb{R}^{5},\qquad
\mathcal{L} = \tfrac1B\sum_b\Big[\,\|a_b - \hat a_b\|_2^2 + \lambda \|h_b\|_1\Big]$$

实现要点：

- **过完备**：$m = 40 \gg d = 5$（也大于真特征数 20，留出冗余——真模型里你不知道特征数）；
- **decoder 归一化**：每步更新后 $W_{d,j} \leftarrow W_{d,j}/\|W_{d,j}\|_2$。否则模型可以"缩小 $h$、放大 $W_d$"把 L1 罚作弊绕过；归一化后 $h_j$ 才有"特征 $j$ 激活强度"的可比语义。代码里 $W_d$ 存为 $(m,d)$ 数组、**行**即字典方向（对应数学记号里 $d\times m$ 矩阵的列），用简化版：只重归一化，不做梯度投影；
- **untied**：$W_e$ 独立于 $W_d$（初始化为 $W_d$ 的拷贝），encoder 还要负责在干涉中"去噪判断"，最优解不是字典的转置 [Bricken 2023]；
- **两个实务 trick**（不加它们你会看到 0.85 左右的"匹配瓶颈"，可以自己关掉试试）：(i) **数据点初始化**——把 $W_d$ 的行初始化为随机非零样本的方向（经典 sparse coding 的字典初始化；本数据大多是单特征射线，等于把字典直接放到真方向附近）；(ii) **学习率衰减**——最后 30% 步数把 lr 降 10 倍，让字典"结晶"到位；
- 反向传播四条链 + 手写 Adam，全 numpy。L1 的次梯度 $\lambda\,\mathrm{sign}(h)$ 进了 ReLU 门之后恒为 $+\lambda$（门开时 $h>0$）：

$$\frac{\partial\mathcal L}{\partial \hat A} = \tfrac{2}{B}(\hat A - A),\qquad
\frac{\partial\mathcal L}{\partial W_d} = H^\top \tfrac{\partial\mathcal L}{\partial \hat A},\qquad
\frac{\partial\mathcal L}{\partial \mathrm{Pre}} = \Big(\tfrac{\partial\mathcal L}{\partial \hat A}\, W_d^\top + \tfrac{\lambda}{B}\Big)\odot \mathbb 1[\mathrm{Pre}>0],\qquad
\frac{\partial\mathcal L}{\partial W_e} = \Big(\tfrac{\partial\mathcal L}{\partial \mathrm{Pre}}\Big)^{\!\top} A$$

练习 2 会让你把这四条链与数值梯度逐元素对拍。

In [ ]:
W_TRUE, P_ACT, M_SAE = W_sparse, 0.05, 40    # 真字典 = 稀疏 toy model 的 W (5,20)

def sample_acts(B, rng, p=P_ACT):
    "SAE 的训练数据：叠加激活 a = W x ∈ R^5（真稀疏码 x 对 SAE 不可见）。"
    return sample_features(B, N_FEAT, p, rng) @ W_TRUE.T

def init_sae(m, rng):
    "数据点初始化：字典方向（W_d 的行）= 随机非零样本的单位方向。"
    A0 = sample_acts(20 * m, rng)
    nz = A0[np.linalg.norm(A0, axis=1) > 0.1]         # 丢掉全零样本
    Wd = nz[rng.choice(len(nz), size=m, replace=False)]
    Wd /= np.linalg.norm(Wd, axis=1, keepdims=True)
    return {"We": Wd.copy(), "be": np.zeros(m),       # untied，但用 W_d 的拷贝初始化
            "Wd": Wd, "bd": np.zeros(D_HID)}

def train_sae(lam, m=M_SAE, steps=12000, batch=1024, lr=1e-3, seed=0):
    rng = np.random.default_rng(seed)
    params = init_sae(m, rng)
    state = adam_init(params)
    for t in range(1, steps + 1):
        lr_t = lr * (0.1 if t > 0.7 * steps else 1.0)    # 末段 lr 衰减，让字典"结晶"
        A = sample_acts(batch, rng)
        Pre  = A @ params["We"].T + params["be"]      # (B,m)
        Hl   = relu(Pre)
        Ahat = Hl @ params["Wd"] + params["bd"]       # (B,d)
        B_ = len(A)
        dAhat = 2 * (Ahat - A) / B_
        dPre  = (dAhat @ params["Wd"].T + lam / B_) * (Pre > 0)   # L1 次梯度被 ReLU 门吸收
        grads = {"Wd": Hl.T @ dAhat, "bd": dAhat.sum(0),
                 "We": dPre.T @ A,   "be": dPre.sum(0)}
        adam_step(params, grads, state, lr_t, t)
        params["Wd"] /= np.linalg.norm(params["Wd"], axis=1, keepdims=True)  # 每步重归一化
    return params

LAM = 0.15
sae = train_sae(LAM)

A_eval = sample_acts(20_000, np.random.default_rng(0))    # 独立评估集
H_eval = relu(A_eval @ sae["We"].T + sae["be"])
Ahat_eval = H_eval @ sae["Wd"] + sae["bd"]
mse  = float(((A_eval - Ahat_eval) ** 2).sum(1).mean())
l1   = float(H_eval.sum(1).mean())
print(f"SAE (λ={LAM})：评估集 MSE={mse:.5f}，平均 ‖h‖₁={l1:.3f}")

## 3 · 验证与度量：字典找回真特征了吗？

沙盒的特权：拿真答案对答案。**匹配矩阵** $C_{ij} = \hat v_i^{\top} \hat W_{d,j}$（被表示的真特征 $i$ × 字典单元 $j$，双方都单位化，内积即余弦）。若 SAE 成功，每个真特征应有一个字典单元与它 $\cos \approx 1$——矩阵呈"每行恰好一个亮点"。注意只对**被 toy model 表示**的特征（$\|W_i\| > 0.5$）做匹配：没被塞进 5 维的特征在激活里根本不存在，SAE 不可能找回。

三个标准度量（讲解页 §4）：

- **L0**：每样本平均激活的字典单元数。本沙盒 ground truth $\approx np = 20 \times 0.05 = 1.0$；
- **重建 $R^2 = 1 - \|A-\hat A\|_F^2 / \|A-\bar A\|_F^2$**：真模型上还要看 loss recovered（重建对下游行为的影响），沙盒里没有"下游"，$R^2$ 即终点；
- **dead features**：评估集上（几乎）从不激活的单元占比——过完备 $m=40$ > 真特征数 20，注定有一批单元失业。

In [ ]:
repr_idx = np.where(norms_sparse > 0.5)[0]                  # 被 toy model 表示的特征
V_true = (W_TRUE[:, repr_idx] / norms_sparse[repr_idx]).T   # (k,5) 单位行 = 真特征方向
C = V_true @ sae["Wd"].T                                    # (k,m) 余弦匹配矩阵
best_cos = C.max(axis=1)
print(f"被表示真特征数 k = {len(repr_idx)}")
print(f"每个真特征的最佳匹配余弦：mean={best_cos.mean():.3f}  min={best_cos.min():.3f}")
assert best_cos.mean() > 0.9, "SAE 字典应与真特征方向高度对齐"

# 可视化：把每行的最佳匹配列排到前面，亮点应连成对角线
matched_cols = list(dict.fromkeys(C.argmax(axis=1)))        # 去重保序
rest = [j for j in range(M_SAE) if j not in set(matched_cols)]
plt.figure(figsize=(9, 3.6))
plt.imshow(C[:, matched_cols + rest], cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="cosine")
plt.axvline(len(matched_cols) - 0.5, color="k", lw=1, ls="--")
plt.xlabel("SAE dictionary unit (matched first, then unused)")
plt.ylabel("true feature")
plt.title("true feature × SAE dictionary cosine (diagonal = one-to-one recovery)")
plt.tight_layout(); plt.show()

In [ ]:
def sae_stats(params, A, act_eps=1e-6, dead_freq=1e-4):
    "返回 (L0, R², dead 单元数, 各单元激活频率)。"
    Hl = relu(A @ params["We"].T + params["be"])
    Ahat = Hl @ params["Wd"] + params["bd"]
    L0 = float((Hl > act_eps).sum(1).mean())
    R2 = float(1 - ((A - Ahat) ** 2).sum() / ((A - A.mean(0)) ** 2).sum())
    freq = (Hl > act_eps).mean(0)
    return L0, R2, int((freq < dead_freq).sum()), freq

L0, R2, n_dead, freq = sae_stats(sae, A_eval)
print(f"L0 = {L0:.2f}（ground truth ≈ {N_FEAT * P_ACT:.1f}）   "
      f"重建 R² = {R2:.4f}   dead = {n_dead}/{M_SAE}")
assert R2 > 0.9

plt.figure(figsize=(8, 2.8))
plt.bar(np.arange(M_SAE), np.sort(freq)[::-1], color="#1f6feb")
plt.yscale("log"); plt.ylim(1e-5, 1)
plt.xlabel("dictionary unit (sorted)"); plt.ylabel("activation freq (log)")
plt.title(f"activation frequency: ~{M_SAE - n_dead} live units vs {n_dead} dead")
plt.tight_layout(); plt.show()
# 两个值得注意的现象（都是真模型上 SAE 的缩影）：
# 1) dead units：m=40 > 真特征数，部分单元失业（练习 4 的 resampling 就是救它们的）；
# 2) L0 略高于 ground truth：活跃单元数 > 真特征数，同一真方向被几个近平行单元分摊
#    ——L1 对"把一次激活拆给两个平行方向"完全无感，这正是 feature splitting 的萌芽。

## 4 · λ 扫描：L0–重建的帕累托前沿

$\lambda$ 控制稀疏–重建的交换率：$\lambda$ 大 → L0 低、可解释性强但重建差（特征被 L1 压死，shrinkage 也更重）；$\lambda$ 小 → 重建好但每样本激活一堆单元、字典方向开始混杂。**不存在唯一正确的 $\lambda$，只有帕累托前沿**——真模型上比较 SAE 架构（ReLU / TopK / JumpReLU / Gated）就是比谁的前沿更靠外（讲解页 §4.2）。在沙盒里有一个特殊参照点：ground truth L0 = 1.0，最优 SAE 应该在 L0≈1 处仍保持高 $R^2$。

In [ ]:
lams = [0.01, 0.03, 0.1, 0.15, 0.3, 0.6]
sweep = []
for lam in lams:
    p_ = train_sae(lam, steps=6000)
    L0_, R2_, nd_, _ = sae_stats(p_, A_eval)
    sweep.append((lam, L0_, R2_, nd_))
    print(f"λ={lam:5.2f}   L0={L0_:6.2f}   R²={R2_:7.4f}   dead={nd_:2d}/{M_SAE}")

xs, ys = [s[1] for s in sweep], [s[2] for s in sweep]
plt.figure(figsize=(6.5, 4))
plt.plot(xs, ys, "o-", color="#1f6feb")
for lam, x, y, _ in sweep:
    plt.annotate(f"λ={lam:g}", (x, y), textcoords="offset points",
                 xytext=(8, -4), fontsize=8)
plt.axvline(N_FEAT * P_ACT, color="gray", ls="--", lw=0.8)
plt.text(N_FEAT * P_ACT + 0.05, min(ys), "true L0", color="gray", fontsize=8)
plt.xlabel("L0 (avg active units / sample)")
plt.ylabel(r"reconstruction $R^2$")
plt.title("L0 vs reconstruction: the sparsity–fidelity Pareto frontier")
plt.tight_layout(); plt.show()

---
## ✏️ 练习 1：实现 SAE 前向 + loss

实现 `sae_forward(A, We, be, Wd, bd, lam)`，返回 `(H, A_hat, loss)`：

$$H = \mathrm{ReLU}(A W_e^\top + b_e),\qquad \hat A = H\, W_d + b_d,\qquad
\mathcal{L} = \tfrac1B\sum_b\Big[\|a_b - \hat a_b\|_2^2 + \lambda\,\|h_b\|_1\Big]$$

约定与正文一致：`A (B,d)`、`We (m,d)`、`Wd (m,d)`（行 = 字典方向）、`H (B,m)`、`loss` 为 float。

**提示**：核心 3 行——`Pre = A @ We.T + be`、`H = np.maximum(Pre, 0.0)`、`A_hat = H @ Wd + bd`；loss 先对特征/维度轴 `sum(axis=1)` 再对 batch `.mean()`（是均值不是求和）；$H \ge 0$ 所以 $\|h\|_1$ 就是 `H.sum(1)`。全函数 5 行以内。

In [ ]:
def sae_forward(A, We, be, Wd, bd, lam):
    # TODO:
    # 1) Pre = A @ We.T + be；H = ReLU(Pre)
    # 2) A_hat = H @ Wd + bd
    # 3) loss = mean_b[ ‖a-â‖² + λ‖h‖₁ ]（先 sum(axis=1) 再 mean）
    # 返回 (H, A_hat, float(loss))
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
We_t = np.array([[1., 0.], [0., 1.], [-1., 0.]])
Wd_t = np.array([[1., 0.], [0., 1.], [1., 0.]])
A_t  = np.array([[2., 3.]])
H_t, Ahat_t, loss_t = sae_forward(A_t, We_t, np.zeros(3), Wd_t, np.zeros(2), lam=0.1)
assert H_t.shape == (1, 3) and Ahat_t.shape == (1, 2)
assert np.allclose(H_t, [[2., 3., 0.]])                  # ReLU 砍掉了 -2
assert np.allclose(Ahat_t, [[2., 3.]])                   # 完美重建
assert abs(loss_t - 0.5) < 1e-12                         # MSE=0，λ·‖h‖₁ = 0.1×5
_, _, loss0 = sae_forward(A_t, We_t, np.zeros(3), Wd_t, np.array([1., -1.]), lam=0.0)
assert abs(loss0 - 2.0) < 1e-12                          # (2-3)² + (3-2)² = 2
_, _, loss2 = sae_forward(np.repeat(A_t, 4, axis=0), We_t, np.zeros(3), Wd_t,
                          np.zeros(2), lam=0.1)
assert abs(loss2 - 0.5) < 1e-12                          # batch 取均值而非求和
print("✅ 练习 1 通过")

## ✏️ 练习 2：手写 SAE 梯度，与数值梯度对拍

实现 `sae_grads(A, We, be, Wd, bd, lam)`，对练习 1 的 loss 求梯度，返回字典 `{"We":…, "be":…, "Wd":…, "bd":…}`（形状与参数一致）。四条链就是 §2 末尾的公式。

**提示**：约 8 行；先重算前向（`Pre, H, A_hat`）；`dAhat = 2*(A_hat - A)/B`；bias 梯度 = 对 batch 轴 `sum(0)`；L1 项的次梯度 $\lambda\,\mathrm{sign}(h)/B$ 在 ReLU 门内恒等于 $\lambda/B$（门开时 $h>0$），所以一行 `dPre = (dAhat @ Wd.T + lam/B) * (Pre > 0)` 同时处理了 MSE 与 L1 两条回传。自测用中心差分逐元素对拍——数值梯度在 ReLU 拐点处会失效，但连续随机数据恰好落在拐点上的概率为 0。

In [ ]:
def sae_grads(A, We, be, Wd, bd, lam):
    # TODO:
    # 前向：Pre, H, A_hat（同练习 1）
    # dAhat = 2 * (A_hat - A) / B
    # dWd = H.T @ dAhat；dbd = dAhat.sum(0)
    # dPre = (dAhat @ Wd.T + lam / B) * (Pre > 0)
    # dWe = dPre.T @ A；dbe = dPre.sum(0)
    # 返回 {"We": dWe, "be": dbe, "Wd": dWd, "bd": dbd}
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rng_g = np.random.default_rng(0)
A_g = rng_g.normal(size=(8, 3))
pg = {"We": rng_g.normal(size=(5, 3)) * 0.7, "be": rng_g.normal(size=5) * 0.3,
      "Wd": rng_g.normal(size=(5, 3)) * 0.7, "bd": rng_g.normal(size=3) * 0.3}
LAM_G = 0.03

def _loss_fn(p):
    Hg = np.maximum(A_g @ p["We"].T + p["be"], 0.0)
    Ag = Hg @ p["Wd"] + p["bd"]
    return float((((A_g - Ag) ** 2).sum(1) + LAM_G * Hg.sum(1)).mean())

g_ana = sae_grads(A_g, pg["We"], pg["be"], pg["Wd"], pg["bd"], LAM_G)
assert set(g_ana) == {"We", "be", "Wd", "bd"}
eps = 1e-5
for k in pg:
    assert g_ana[k].shape == pg[k].shape
    g_num = np.zeros_like(pg[k])
    flat, nflat = pg[k].reshape(-1), g_num.reshape(-1)
    for i in range(flat.size):
        orig = flat[i]
        flat[i] = orig + eps; lp = _loss_fn(pg)
        flat[i] = orig - eps; lm = _loss_fn(pg)
        flat[i] = orig
        nflat[i] = (lp - lm) / (2 * eps)
    err = np.abs(g_ana[k] - g_num).max()
    assert err < 1e-6, f"梯度 {k} 与数值梯度不符（max err = {err:.2e}）"
print("✅ 练习 2 通过")

## ✏️ 练习 3：贪心特征–字典匹配

正文 §3 的"每行取 max"允许两个真特征匹配到同一个字典单元——验证"一一对应"需要不放回的匹配。实现 `greedy_match(V, D)`：`V (k,d)` 真特征单位方向、`D (m,d)` 字典单位方向（$m \ge k$）。反复取余弦矩阵的**全局最大**配对，配对后划掉该行该列，保证一对一。返回 `(matches, coses)`：`matches[i]` = 特征 $i$ 配到的字典行号（int 数组，互不重复），`coses[i]` = 对应余弦。

**提示**：`C = V @ D.T`（双方已单位化，内积即余弦）；在副本 `Cw` 上循环 k 次：`i, j = np.unravel_index(np.argmax(Cw), Cw.shape)`，记录 `C[i, j]` 后 `Cw[i, :] = -np.inf; Cw[:, j] = -np.inf`；约 10 行。贪心不保证全局最优（那要匈牙利算法），但匹配矩阵接近"每行一个亮点"时两者一致。

In [ ]:
def greedy_match(V, D):
    # TODO:
    # 1) C = V @ D.T；Cw = C.copy()
    # 2) 循环 k 次：全局 argmax → (i, j)；matches[i] = j；coses[i] = C[i, j]
    #    然后 Cw[i, :] = -np.inf；Cw[:, j] = -np.inf
    # 3) 返回 (matches: int 数组 (k,), coses: float 数组 (k,))
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng_m = np.random.default_rng(0)
V_toy = rng_m.normal(size=(4, 6)); V_toy /= np.linalg.norm(V_toy, axis=1, keepdims=True)
noise = rng_m.normal(size=(3, 6)); noise /= np.linalg.norm(noise, axis=1, keepdims=True)
D_toy = np.vstack([V_toy[[2, 0, 3, 1]], noise])      # 字典 = 打乱的真方向 + 3 个噪声行
m_toy, c_toy = greedy_match(V_toy, D_toy)
assert list(m_toy) == [1, 3, 0, 2]                   # 精确还原置换
assert np.all(c_toy > 0.999)
assert len(set(np.asarray(m_toy).tolist())) == len(m_toy)   # 一对一

m_sae, c_sae = greedy_match(V_true, sae["Wd"])       # 真特征 vs 训练好的 SAE 字典
assert len(set(np.asarray(m_sae).tolist())) == len(m_sae)
assert c_sae.mean() > 0.9, f"一对一平均余弦 {c_sae.mean():.3f}，应 > 0.9"
print(f"SAE 一对一匹配：平均余弦 = {c_sae.mean():.3f}，最小 = {c_sae.min():.3f}")
print("✅ 练习 3 通过")

## ✏️ 练习 4：dead feature 检测与 resampling

[Bricken 2023] resampling 的最小版（讲解页 §4.1）：dead 单元的 ReLU 梯度恒为零、自己救不回来，于是定期把它"转岗"到模型最解释不了的数据方向上。实现两个函数：

1. `detect_dead(H, freq_thresh=1e-3)`：`H (B,m)` 为一批隐层激活；单元 $j$ 的激活频率 = `(H[:, j] > 1e-6)` 沿 batch 的均值；返回布尔数组 `(m,)`，频率 < `freq_thresh` 记为 dead。
2. `resample_dead(We, be, Wd, A, A_hat, dead, scale=0.02)`：把第 $k$ 个 dead 单元（按下标升序）的 decoder 行设为**重建误差第 $k$ 大**的样本方向（单位化），encoder 行 = 同方向 × `scale`，encoder bias 置 0；**返回新的 `(We, be, Wd)`，不要原地修改**。

**提示**：误差 `err = ((A - A_hat)**2).sum(1)`；`worst = np.argsort(err)[::-1]`；`for k, j in enumerate(np.where(dead)[0])` 把 dead 单元与最差样本配对；先 `We.copy()` 等再改；约 12 行。可假设 dead 单元数 ≤ 样本数且对应样本非零。

In [ ]:
def detect_dead(H, freq_thresh=1e-3):
    # TODO: freq = (H > 1e-6).mean(axis=0)；返回 freq < freq_thresh（布尔数组）
    raise NotImplementedError

def resample_dead(We, be, Wd, A, A_hat, dead, scale=0.02):
    # TODO:
    # 1) 复制 We, be, Wd（不要原地修改）
    # 2) err = ((A - A_hat)**2).sum(1)；worst = np.argsort(err)[::-1]
    # 3) 第 k 个 dead 单元 j：v = A[worst[k]] 单位化；Wd[j] = v；We[j] = scale*v；be[j] = 0
    # 4) 返回 (We_new, be_new, Wd_new)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
H_t4 = np.zeros((200, 6))
H_t4[:, 0] = 1.0                       # 频率 1.0
H_t4[::2, 1] = 0.5                     # 频率 0.5
H_t4[0, 4] = 0.3                       # 频率 1/200 = 0.005 ≥ 1e-3 → 不算 dead
dead_t = detect_dead(H_t4)
assert dead_t.dtype == bool and dead_t.shape == (6,)
assert list(dead_t) == [False, False, True, True, False, True]

We_o = np.ones((6, 3)); be_o = np.ones(6); Wd_o = np.tile([1., 0., 0.], (6, 1))
A_o    = np.array([[0., 0., 2.], [0., 3., 0.], [1., 0., 0.], [0.5, 0., 0.]])
Ahat_o = np.zeros((4, 3))              # err = [4, 9, 1, 0.25] → 最差顺序 1, 0, 2, 3
We_n, be_n, Wd_n = resample_dead(We_o, be_o, Wd_o, A_o, Ahat_o, dead_t)
assert np.allclose(Wd_n[2], [0., 1., 0.])              # dead#0 ← 误差最大样本 A[1] 方向
assert np.allclose(Wd_n[3], [0., 0., 1.])              # dead#1 ← 第二大 A[0]
assert np.allclose(Wd_n[5], [1., 0., 0.])              # dead#2 ← 第三大 A[2]
assert np.allclose(We_n[2], 0.02 * Wd_n[2]) and be_n[2] == 0.0
assert np.allclose(Wd_n[0], Wd_o[0]) and np.allclose(We_n[0], 1.0) and be_n[0] == 1.0
assert np.allclose(We_o, 1.0) and np.allclose(be_o, 1.0)   # 原数组未被修改
print("✅ 练习 4 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def sae_forward(A, We, be, Wd, bd, lam):
    Pre = A @ We.T + be
    H = np.maximum(Pre, 0.0)
    A_hat = H @ Wd + bd
    loss = float((((A - A_hat) ** 2).sum(1) + lam * H.sum(1)).mean())
    return H, A_hat, loss

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def sae_grads(A, We, be, Wd, bd, lam):
    B = len(A)
    Pre = A @ We.T + be
    H = np.maximum(Pre, 0.0)
    A_hat = H @ Wd + bd
    dAhat = 2 * (A_hat - A) / B
    dPre = (dAhat @ Wd.T + lam / B) * (Pre > 0)
    return {"We": dPre.T @ A, "be": dPre.sum(0),
            "Wd": H.T @ dAhat, "bd": dAhat.sum(0)}

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def greedy_match(V, D):
    C = V @ D.T
    Cw = C.copy()
    k = V.shape[0]
    matches = np.full(k, -1, dtype=int)
    coses = np.zeros(k)
    for _ in range(k):
        i, j = np.unravel_index(np.argmax(Cw), Cw.shape)
        matches[i], coses[i] = j, C[i, j]
        Cw[i, :] = -np.inf
        Cw[:, j] = -np.inf
    return matches, coses

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def detect_dead(H, freq_thresh=1e-3):
    return (H > 1e-6).mean(axis=0) < freq_thresh

def resample_dead(We, be, Wd, A, A_hat, dead, scale=0.02):
    We_n, be_n, Wd_n = We.copy(), be.copy(), Wd.copy()
    err = ((A - A_hat) ** 2).sum(1)
    worst = np.argsort(err)[::-1]
    for k, j in enumerate(np.where(dead)[0]):
        v = A[worst[k]] / np.linalg.norm(A[worst[k]])
        Wd_n[j], We_n[j], be_n[j] = v, scale * v, 0.0
    return We_n, be_n, Wd_n

---
## 🎯 真实数据胶囊题：真实 GPT-2 embedding 上训一个迷你 SAE

SAE 用稀疏字典从激活里拆出可解释特征。在真实 GPT-2 embedding 上训一个单隐层 SAE(带 L1 稀疏)，验证它能低误差重构、且隐藏激活是稀疏的。

> 本模块新增的**真实数据**练习：用**真实 GPT-2 权重/embedding**把本章的可解释性技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, struct, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.interp_data"); os.makedirs(CACHE,exist_ok=True)
ST="https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"
def _rng(s,e):
    req=urllib.request.Request(ST, headers={"Range":f"bytes={s}-{e}"})
    return urllib.request.urlopen(req,timeout=60).read()
def gpt2_emb_block(n=6000):
    cache=os.path.join(CACHE,f"wte_{n}.npy")
    if os.path.exists(cache): return np.load(cache)
    hlen=struct.unpack("<Q", _rng(0,7))[0]; hdr=json.loads(_rng(8,8+hlen-1))
    info=hdr["wte.weight"]; base=8+hlen; s0=info["data_offsets"][0]; d=info["shape"][1]
    raw=_rng(base+s0, base+s0+n*d*4-1)
    E=np.frombuffer(raw,dtype=np.float32).reshape(n,d).copy()
    np.save(cache,E); return E
def gpt2_vocab():
    p=os.path.join(CACHE,"vocab.json")
    if not os.path.exists(p): urllib.request.urlretrieve("https://huggingface.co/openai-community/gpt2/resolve/main/vocab.json",p)
    return json.load(open(p))
def digit_letter_dataset(lim=6000):
    "返回 (X[token嵌入], y[1=数字 0=字母], E, ids_digit, ids_alpha)"
    v=gpt2_vocab(); E=gpt2_emb_block(lim)
    dig=[i for t,i in v.items() if i<lim and t.isdigit()]
    alpha=[i for t,i in v.items() if i<lim and t.isalpha() and t.isascii()]
    rng=np.random.default_rng(0); alpha=list(rng.permutation(alpha)[:len(dig)])
    ids=dig+alpha; y=np.array([1]*len(dig)+[0]*len(alpha))
    return E[ids], y, E, dig, alpha
def shakespeare():
    p=os.path.join(CACHE,"shake.txt")
    if not os.path.exists(p): urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",p)
    return open(p).read()

E=gpt2_emb_block(6000)
X=(E - E.mean(0))/(E.std(0)+1e-8)   # 标准化真实 embedding
X=X[:2000]
print("训练数据(真实标准化 embedding):", X.shape)

**练习**：实现 `sae_step(X, W_enc, W_dec, b, l1, lr)`：encode `h=relu(X@W_enc+b)`、decode `X̂=h@W_dec`，loss=重构 MSE + l1·|h|，做一步梯度更新，返回 `(W_enc,W_dec,b, 重构误差, 平均稀疏度)`。

In [ ]:
def sae_step(X, W_enc, W_dec, b, l1=1e-3, lr=0.01):
    # TODO: 前向 h=relu(X@W_enc+b), Xhat=h@W_dec; 反向(MSE+L1)更新; 返回新参数+recon误差+稀疏度(h>0比例)
    raise NotImplementedError


In [ ]:
# 自测：多步后重构误差下降
rng=np.random.default_rng(0); d=X.shape[1]; m=256
W_enc=rng.normal(0,0.05,(d,m)); W_dec=rng.normal(0,0.05,(m,d)); b=np.zeros(m)
err0=None
for t in range(150):
    W_enc,W_dec,b,err,spars=sae_step(X,W_enc,W_dec,b,l1=1e-3,lr=0.02)
    if t==0: err0=err
assert err < err0, f"SAE 重构误差应下降 {err0:.3f}->{err:.3f}"
assert spars < 1.0, "应有一定稀疏度(部分隐藏单元关闭)"
print(f"SAE ✓  重构误差 {err0:.3f}->{err:.3f}, 激活稀疏度(激活比例)={spars:.2f}")


### 📖 参考答案

In [ ]:
def sae_step(X, W_enc, W_dec, b, l1=1e-3, lr=0.01):
    z=X@W_enc+b; h=np.maximum(0,z); Xhat=h@W_dec
    err=np.mean((Xhat-X)**2)
    dXhat=2*(Xhat-X)/X.shape[0]
    dW_dec=h.T@dXhat
    dh=dXhat@W_dec.T + l1*np.sign(h)
    dz=dh*(z>0)
    dW_enc=X.T@dz; db=dz.sum(0)
    W_enc-=lr*dW_enc; W_dec-=lr*dW_dec; b-=lr*db
    return W_enc,W_dec,b,float(err),float((h>0).mean())
print("✓ SAE = 稀疏字典学习，从叠加(superposition)的激活里拆出单义特征")